In [ ]:
import os
import re
import sys
from pathlib import Path
import torch
import pandas as pd
import csv
from typing import List, Dict, Optional
from huggingface_hub import HfApi, snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM

# ================= CONFIG =================
DATA_DIR = r"D:\Dementia-Thesis - Don't access without permission\cookie_control_dementia"
MODEL_NAME = "gpt-oss:20b"
HF_TOKEN = os.environ.get("HF_TOKEN")
OUTPUT_CSV = r"D:\Dementia-Thesis - Don't access without permission\Thesis\gpt_oss_cot_multiclass.csv"

# Generation parameters
GEN_MAX_NEW_TOKENS = 1024  # Increased for CoT reasoning
GEN_TEMPERATURE = 0.3
GEN_TOP_P = 0.9

# ================= HF SANITY CHECK =================
def hf_sanity_check(model_name: str, token: Optional[str] = None) -> bool:
    """Verify model is accessible on HuggingFace Hub."""
    if "gpt-oss" in model_name.lower():
        print(f"✅ Using local/Ollama model: {model_name}")
        return True
    
    api = HfApi()
    try:
        info = api.model_info(model_name, token=token)
        print(f"HF CHECK OK: {info.modelId} (private={info.private})")
        return True
    except Exception as e:
        print(f"HF CHECK ERROR: {type(e).__name__}: {e}", file=sys.stderr)
        return False
    
# ================= TOKENIZER / MODEL LOADERS =================
def load_gpt_oss_tokenizer(model_name: str, hf_token: Optional[str] = None):
    """Load tokenizer for gpt-oss model."""
    try:
        # Use GPT-2 tokenizer as base for gpt-oss compatibility
        tok = AutoTokenizer.from_pretrained("gpt2", trust_remote_code=True)
    except Exception as e:
        print(f"[Tokenizer] Failed to load: {e}", file=sys.stderr)
        from transformers import GPT2Tokenizer
        tok = GPT2Tokenizer.from_pretrained("gpt2")
    
    # Ensure chat template exists
    if not getattr(tok, "chat_template", None):
        tok.chat_template = (
            "{% for m in messages %}"
            "{{ m['role'] }}: {{ m['content'] }}\n"
            "{% endfor %}"
            "assistant:"
        )
    return tok

def load_gpt_oss_model(model_name: str, hf_token: Optional[str] = None, 
                       torch_dtype=torch.float16):
    """Load gpt-oss model from local/Ollama or fallback to distilgpt2."""
    print(f"[Model] Model identifier: {model_name}")
    
    # Check if this is an Ollama format (gpt-oss:20b)
    if ":" in model_name and "gpt-oss" in model_name.lower():
        print(f"[Model] Detected Ollama/local format: {model_name}")
        print(f"[Model] NOTE: Using distilgpt2 as fallback (Ollama requires separate setup)")
        # Fall through to use fallback
    
    # Try truly open models without auth requirements
    fallbacks = [
        "distilgpt2",  # Small, no auth
        "gpt2",        # Classic, no auth
    ]
    
    for fallback in fallbacks:
        try:
            print(f"[Model] Attempting to load {fallback}...", file=sys.stderr)
            return AutoModelForCausalLM.from_pretrained(
                fallback, 
                trust_remote_code=True, 
                torch_dtype=torch_dtype, 
                device_map="auto"
            )
        except Exception as e:
            print(f"[Model] {fallback} failed: {type(e).__name__}", file=sys.stderr)
            continue
    
    # If all fallbacks fail
    raise OSError(
        f"Could not load {model_name} or any fallback model.\n"
        "If using Ollama (gpt-oss:20b), ensure:\n"
        "  1. Ollama is running: 'ollama serve'\n"
        "  2. Model is pulled: 'ollama pull gpt-oss:20b'\n"
        "  3. Use ollama_client_generate() instead of model.generate()"
    )
    
# ================= PROMPT / PARSING =================
def extract_patient_info(filename: str):
    """Extract patient ID from filename (first 3 digits)."""
    match = re.match(r'(\d{3})', filename)
    return match.group(1) if match else None

def create_cot_prompt(filename: str, transcript: str) -> str:
    """Create a Chain-of-Thought prompt for dementia diagnosis."""
    return f"""You are an expert neuropsychologist specializing in dementia assessment. Analyze the following Cookie Theft picture description transcript.

**File:** {filename}

**Transcript:**
{transcript}

**Task:**
Analyze this transcript step by step to determine if this person shows signs of cognitive impairment.

**Think through the following aspects:**

1. **Language Fluency**: Is the speech fluent or hesitant? Are there many pauses, fillers, or restarts?

2. **Vocabulary & Word Finding**: Does the speaker use specific words or vague terms? Are there word-finding difficulties or circumlocutions?

3. **Content & Completeness**: Does the speaker describe the main elements of the Cookie Theft picture (mother, children, stool, cookies, sink overflow, window)?

4. **Grammar & Syntax**: Are sentences well-formed or fragmented? Is there evidence of grammatical errors?

5. **Coherence & Organization**: Is the description organized and logical, or scattered and disjointed?

6. **Repetitions**: Are there unnecessary repetitions of words, phrases, or ideas?

**Step-by-step reasoning:**
[Provide your detailed analysis for each aspect above]

**Final Assessment:**
Based on your step-by-step analysis above, provide your prediction:

Label: [Control/MCI/ProbableAD]
MMSE: [0-30]
Confidence: [Low/Medium/High]"""

def parse_llm_response(response: str, filename: str = "") -> Dict:
    """Parse LLM response to extract label, MMSE, and confidence."""
    mmse_match = re.search(r'MMSE:\s*(\d+)', response, re.IGNORECASE)
    conf_match = re.search(r'Confidence:\s*(\w+)', response, re.IGNORECASE)
    
    # More flexible matching for Control, MCI, or ProbableAD
    label = 'Unknown'
    response_lower = response.lower()
    
    # Look for final assessment section first
    final_section = response_lower.split('final assessment')[-1] if 'final assessment' in response_lower else response_lower
    
    # Check for label variations (Control, MCI, ProbableAD)
    if re.search(r'label:\s*probable\s*ad', final_section) or re.search(r'label:\s*probablead', final_section):
        label = 'ProbableAD'
    elif re.search(r'label:\s*mci', final_section):
        label = 'MCI'
    elif re.search(r'label:\s*control', final_section):
        label = 'Control'
    # Fallback: check entire response
    elif re.search(r'probable\s*ad', response_lower) or 'probablead' in response_lower:
        label = 'ProbableAD'
    elif re.search(r'\bmci\b', response_lower):
        label = 'MCI'
    elif re.search(r'\bcontrol\b', response_lower):
        label = 'Control'
    elif re.search(r'\bdementia\b|\balzheimer', response_lower):
        label = 'ProbableAD'
    elif re.search(r'\bmild\s*cognitive', response_lower):
        label = 'MCI'
    elif re.search(r'\bnormal\b|\bhealthy\b', response_lower):
        label = 'Control'
    
    if label == 'Unknown':
        print(f"    [DEBUG] Could not parse label from response")
        print(f"    [DEBUG] Raw response preview (first 500 chars):")
        print(f"    {response[:500]}")
        print(f"    [DEBUG] Response length: {len(response)} chars")
        if filename:
            debug_file = Path(OUTPUT_CSV).parent / f"debug_response_{filename.replace('.txt', '')}.txt"
            with open(debug_file, 'w', encoding='utf-8') as f:
                f.write(response)
            print(f"    [DEBUG] Full response saved to: {debug_file}")
    
    return {
        'label': label,
        'mmse': int(mmse_match.group(1)) if mmse_match else -1,
        'confidence': conf_match.group(1) if conf_match else 'Unknown',
        'raw': response
    }

# ================= CHAT / GENERATION =================
def render_messages_to_text(messages: List[Dict], tokenizer) -> str:
    """Render messages to text format for model input."""
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    
    # Fallback: manual formatting
    out = ""
    for m in messages:
        role = m.get("role", "")
        content = m.get("content", "")
        out += f"{role}: {content}\n"
    out += "assistant:"
    return out

def ollama_generate(prompt: str, model_name: str = "gpt-oss:20b") -> str:
    """Generate text using Ollama (for local gpt-oss:20b)."""
    try:
        import requests
    except ImportError:
        raise ImportError("requests library required for Ollama. Install: pip install requests")
    
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model_name,
                "prompt": prompt,
                "stream": False,
                "temperature": GEN_TEMPERATURE,
            },
            timeout=300
        )
        response.raise_for_status()
        return response.json()["response"]
    except requests.exceptions.ConnectionError:
        raise ConnectionError(
            f"Cannot connect to Ollama at http://localhost:11434\n"
            f"Ensure Ollama is running: ollama serve"
        )
    except Exception as e:
        raise RuntimeError(f"Ollama generation failed: {e}")

def generate_from_model(text: str, tokenizer, model, use_ollama: bool = False, model_name: str = "gpt-oss:20b") -> str:
    """Generate text from model given prompt."""
    if use_ollama:
        return ollama_generate(text, model_name)
    
    model_inputs = tokenizer(
        [text], 
        return_tensors="pt", 
        truncation=True, 
        padding=True
    ).to(next(model.parameters()).device)
    
    generated = model.generate(
        input_ids=model_inputs.input_ids,
        attention_mask=model_inputs.attention_mask,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        do_sample=True,
        pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id else tokenizer.eos_token_id)
    )
    
    for in_ids, out_ids in zip(model_inputs.input_ids, generated):
        new_ids = out_ids[len(in_ids):]
        return tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    return ""

# ================= COT PREDICTION =================
def predict_visit_cot(filename: str, transcript: str, tokenizer, model, 
                      use_ollama: bool = False, model_name: str = "gpt-oss:20b") -> str:
    """
    Make prediction using Chain-of-Thought prompting.
    The model reasons step-by-step before giving the final answer.
    """
    main_prompt = create_cot_prompt(filename, transcript)
    
    messages = [
        {"role": "system", "content": "You are an expert neuropsychologist diagnosing cognitive impairment from speech."},
        {"role": "user", "content": main_prompt}
    ]
    
    text = render_messages_to_text(messages, tokenizer)
    return generate_from_model(text, tokenizer, model, use_ollama=use_ollama, model_name=model_name)

# ================= MAIN EXECUTION =================
def main():
    """Main execution: load model, process transcripts, save predictions."""
    
    # Determine if using Ollama
    use_ollama = "gpt-oss" in MODEL_NAME.lower() and ":" in MODEL_NAME
    
    # Sanity check
    print("=" * 70)
    print("CHECKING MODEL...")
    print("=" * 70)
    hf_sanity_check(MODEL_NAME, HF_TOKEN)
    
    # Load model and tokenizer
    print("\n" + "=" * 70)
    print("LOADING TOKENIZER AND MODEL...")
    print("=" * 70)
    tokenizer = load_gpt_oss_tokenizer(MODEL_NAME, hf_token=HF_TOKEN)
    
    if use_ollama:
        print(f"ℹ️  Using Ollama for model: {MODEL_NAME}")
        print(f"⚠️  Ensure Ollama is running: ollama serve")
        model = None  # Not needed for Ollama
        try:
            # Test Ollama connection
            _ = ollama_generate("test")
            print("✅ Ollama connection successful")
        except Exception as e:
            print(f"❌ Ollama test failed: {e}", file=sys.stderr)
            print("Falling back to local distilgpt2 model...")
            use_ollama = False
            model = load_gpt_oss_model(
                MODEL_NAME, 
                hf_token=HF_TOKEN,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
            )
    else:
        model = load_gpt_oss_model(
            MODEL_NAME, 
            hf_token=HF_TOKEN,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )
        print("✅ Model loaded successfully")
        if model:
            print(f"   Device: {next(model.parameters()).device}")
    
    # Get transcript files
    files = sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".txt"))
    print(f"\n✅ Found {len(files)} transcript files in {DATA_DIR}")
    
    # Process each transcript
    print("\n" + "=" * 70)
    print("PROCESSING TRANSCRIPTS (CHAIN-OF-THOUGHT)...")
    print("=" * 70)
    
    results = []
    for i, filename in enumerate(files, 1):
        pid = extract_patient_info(filename)
        path = Path(DATA_DIR) / filename
        
        print(f"\n[{i:3d}/{len(files)}] Processing {filename} (Patient ID: {pid})")
        
        try:
            with open(path, "r", encoding="utf-8") as f:
                transcript = f.read().strip()
            
            if not transcript:
                raise ValueError("Empty transcript")
            
            # Make prediction using Chain-of-Thought
            response = predict_visit_cot(
                filename, transcript, tokenizer, model, 
                use_ollama=use_ollama, model_name=MODEL_NAME
            )
            prediction = parse_llm_response(response, filename)
            
            print(f"           → Label: {prediction['label']:12s} | MMSE: {prediction['mmse']:3d} | Confidence: {prediction['confidence']}")
            
            # Store result (remove .txt from filename)
            file_id = re.sub(r'\.txt$', '', filename)
            results.append({
                "id": file_id,
                "confidence": prediction["confidence"],
                "mmse": prediction["mmse"],
                "dx": prediction["label"]
            })
            
        except Exception as e:
            print(f"           ⚠️  ERROR: {e}", file=sys.stderr)
            file_id = re.sub(r'\.txt$', '', filename)
            results.append({
                "id": file_id,
                "confidence": "N/A",
                "mmse": -1,
                "dx": "Error"
            })
    
    # Save results
    print("\n" + "=" * 70)
    print("SAVING RESULTS...")
    print("=" * 70)
    df = pd.DataFrame(results, columns=["id", "confidence", "mmse", "dx"])
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Results saved to: {OUTPUT_CSV}")
    
    # Display sample
    print("\n" + "=" * 70)
    print("SAMPLE PREDICTIONS (first 10 rows):")
    print("=" * 70)
    print(df.head(10).to_string(index=False))
    
    print(f"\n✅ Processing complete! Total: {len(results)} predictions")
    
    return df

if __name__ == "__main__":
    df_results = main()

In [ ]:
# ================= EVALUATION: COT PREDICTIONS VS ACTUAL LABELS =================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

# ---------- Paths ----------
actual_path = r"D:\Dementia-Thesis - Don't access without permission\Thesis\3_classes.csv"
pred_path   = r"D:\Dementia-Thesis - Don't access without permission\Thesis\gpt_oss_cot_multiclass.csv"

# ---------- Load CSVs ----------
print("=" * 70)
print("LOADING CSVs...")
print("=" * 70)
df_true = pd.read_csv(actual_path)
df_pred = pd.read_csv(pred_path)
print(f"✅ Loaded actual labels: {actual_path}")
print(f"✅ Loaded predictions: {pred_path}")
print(f"   Actual shape: {df_true.shape}")
print(f"   Predictions shape: {df_pred.shape}")

# ---------- Column mapping ----------
# Actual CSV columns: id, mmse, dx (or similar)
# Prediction CSV columns: id, confidence, mmse, dx

# For actual data - use mmse and dx columns
mmse_true_col = "mmse"
label_true_col = "dx"

# For prediction data - use mmse and dx columns
mmse_pred_col = "mmse"
label_pred_col = "dx"

print(f"\nUsing columns:")
print(f"  Actual   → mmse: '{mmse_true_col}' | label: '{label_true_col}'")
print(f"  Predicted → mmse: '{mmse_pred_col}' | label: '{label_pred_col}'")

# ---------- Rename for consistency ----------
df_true = df_true.rename(columns={mmse_true_col: "actual_mmse", label_true_col: "actual_label"})
df_pred = df_pred.rename(columns={mmse_pred_col: "pred_mmse", label_pred_col: "pred_label"})

# ---------- Ensure 'id' exists in both ----------
if "id" not in df_true.columns or "id" not in df_pred.columns:
    raise KeyError("Both CSVs must contain an 'id' column for matching (patient id).")

# ---------- Merge on id ----------
df_merged = pd.merge(df_pred, df_true, on="id", how="inner", suffixes=("_pred", "_true"))
if df_merged.empty:
    raise ValueError("Merge resulted in 0 rows. Check that 'id' values match between files.")

print(f"\n✅ Merged {len(df_merged)} rows on 'id'")

# ---------- Normalize label strings ----------
df_merged["pred_label_norm"] = df_merged["pred_label"].astype(str).str.strip().str.lower()
df_merged["actual_label_norm"] = df_merged["actual_label"].astype(str).str.strip().str.lower()
df_merged["label_match"] = df_merged["pred_label_norm"] == df_merged["actual_label_norm"]

# ---------- Numeric MMSE ----------
df_merged["pred_mmse"] = pd.to_numeric(df_merged["pred_mmse"], errors="coerce")
df_merged["actual_mmse"] = pd.to_numeric(df_merged["actual_mmse"], errors="coerce")
df_merged["mmse_diff"] = (df_merged["pred_mmse"] - df_merged["actual_mmse"]).abs()

# ---------- Metrics ----------
label_accuracy = df_merged["label_match"].mean() * 100
mae = df_merged["mmse_diff"].mean()
corr = df_merged["pred_mmse"].corr(df_merged["actual_mmse"])

# Optional: per-class precision/recall/f1
labels = sorted(set(df_merged["actual_label_norm"].unique()) | set(df_merged["pred_label_norm"].unique()))
y_true = df_merged["actual_label_norm"]
y_pred = df_merged["pred_label_norm"]
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)

prf_df = pd.DataFrame({
    "label": labels,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support
})

# ---------- Print summary ----------
print("\n" + "=" * 70)
print("📊 EVALUATION RESULTS (GPT-OSS:20B Chain-of-Thought)")
print("=" * 70)
print(f"Samples evaluated: {len(df_merged)}")
print(f"Label Accuracy: {label_accuracy:.2f}% ({int(df_merged['label_match'].sum())}/{len(df_merged)})")
print(f"MMSE Mean Absolute Error (MAE): {mae:.3f}")
print(f"MMSE Pearson correlation: {corr:.3f}\n")

print("Per-class Precision/Recall/F1:")
print("=" * 70)
print(prf_df.to_string(index=False))

# ---------- Save merged CSV ----------
out_csv = r"D:\Dementia-Thesis - Don't access without permission\Thesis\gpt_oss_cot_multiclass_vs_actual_evaluation.csv"
df_merged.to_csv(out_csv, index=False)
print(f"\n✅ Saved merged evaluation CSV: {out_csv}")

# ---------- Confusion matrix ----------
cm = confusion_matrix(df_merged["actual_label_norm"], df_merged["pred_label_norm"], labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)

plt.figure(figsize=(8,6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar_kws={"label": "Count"})
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("Actual Label", fontsize=12)
plt.title("Confusion Matrix - GPT-OSS:20B COT Multi-Class Classification")
confusion_path = r"D:\Dementia-Thesis - Don't access without permission\Thesis\gpt_oss_cot_multiclass_confusion_matrix.png"
plt.savefig(confusion_path, bbox_inches="tight", dpi=300)
plt.close()
print(f"✅ Saved confusion matrix to: {confusion_path}")

# ---------- Scatter plot: actual vs predicted MMSE ----------
plt.figure(figsize=(6,6))
plt.scatter(df_merged["actual_mmse"], df_merged["pred_mmse"], alpha=0.7, edgecolors='k', linewidths=0.5, s=100)
xmin = min(df_merged["actual_mmse"].min(), df_merged["pred_mmse"].min())
xmax = max(df_merged["actual_mmse"].max(), df_merged["pred_mmse"].max())
plt.plot([xmin, xmax], [xmin, xmax], linestyle="--", color="red", linewidth=2, label="Perfect prediction")
plt.xlabel("Actual MMSE", fontsize=12)
plt.ylabel("Predicted MMSE", fontsize=12)
plt.title(f"Predicted vs Actual MMSE (r={corr:.3f})", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
scatter_path = r"D:\Dementia-Thesis - Don't access without permission\Thesis\gpt_oss_cot_multiclass_mmse_scatter.png"
plt.savefig(scatter_path, bbox_inches="tight", dpi=300)
plt.close()
print(f"✅ Saved MMSE scatter plot to: {scatter_path}")

# ---------- Save per-class PRF ----------
prf_out = r"D:\Dementia-Thesis - Don't access without permission\Thesis\gpt_oss_cot_multiclass_per_class_prf.csv"
prf_df.to_csv(prf_out, index=False)
print(f"✅ Saved per-class precision/recall/f1 to: {prf_out}")

# ---------- Small summary file ----------
summary = {
    "n_samples": [len(df_merged)],
    "label_accuracy_pct": [label_accuracy],
    "mmse_mae": [mae],
    "mmse_corr": [corr]
}
summary_path = r"D:\Dementia-Thesis - Don't access without permission\Thesis\gpt_oss_cot_multiclass_eval_summary.csv"
pd.DataFrame(summary).to_csv(summary_path, index=False)
print(f"✅ Saved eval summary to: {summary_path}")

print("\n" + "=" * 70)
print("✅ EVALUATION COMPLETE!")
print("=" * 70)